In [3]:
import sys
import os

# --- Check where Python is looking ---
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
print("Notebook is in  :", os.getcwd())
print("Project root    :", project_root)
print("Looking for file:", os.path.join(project_root, 'ab_test_framework.py'))
print("File exists?    :", os.path.exists(os.path.join(project_root, 'ab_test_framework.py')))

# --- List all files in project root ---
print("\nFiles in project root:")
for f in os.listdir(project_root):
    print(" ", f)

Notebook is in  : C:\Users\R D\AB_Test_FrameWork\notebooks
Project root    : C:\Users\R D\AB_Test_FrameWork
Looking for file: C:\Users\R D\AB_Test_FrameWork\ab_test_framework.py
File exists?    : False

Files in project root:
  .ipynb_checkpoints
  ab_test_framework.py.py
  cookie_cats.csv
  data
  notebooks
  sql


In [4]:
print("File exists?    :", os.path.exists(os.path.join(project_root, 'ab_test_framework.py')))

print("\nFiles in project root:")
for f in os.listdir(project_root):
    print(" ", f)

File exists?    : True

Files in project root:
  .ipynb_checkpoints
  ab_test_framework.py
  cookie_cats.csv
  data
  notebooks
  sql


In [10]:
import sys

# --- Add project root to Python path ---
sys.path.insert(0, project_root)

# --- Import the framework ---
from ab_test_framework import *

print("Project root:", project_root)

Project root: C:\Users\R D\AB_Test_FrameWork


In [11]:
import pandas as pd
import os

# --- Load cleaned data ---
cleaned_path = os.path.join(project_root, 'data', 'cookie_cats_cleaned.csv')
df = pd.read_csv(cleaned_path)

# --- Test outcome type detection ---
print("=== Outcome Type Detection ===")
for col in ['sum_gamerounds', 'retention_1', 'retention_7']:
    result = detect_outcome_type(df[col])
    print(f"  {col:20s} dtype={str(df[col].dtype):8s} → {result}")

=== Outcome Type Detection ===
  sum_gamerounds       dtype=int64    → continuous
  retention_1          dtype=bool     → categorical
  retention_7          dtype=bool     → categorical


In [13]:
# --- Reload framework to pick up new functions ---
import importlib
import ab_test_framework
importlib.reload(ab_test_framework)
from ab_test_framework import *

# --- Test normality check on both groups ---
gate_30 = df[df['version'] == 'gate_30']['sum_gamerounds']
gate_40 = df[df['version'] == 'gate_40']['sum_gamerounds']

print("=== Normality Check: sum_gamerounds ===")
print("\ngate_30:")
norm_30 = check_normality(gate_30)
for k, v in norm_30.items():
    print(f"  {k:12s}: {v}")

print("\ngate_40:")
norm_40 = check_normality(gate_40)
for k, v in norm_40.items():
    print(f"  {k:12s}: {v}")
# --- Test normality check on both groups ---
gate_30 = df[df['version'] == 'gate_30']['sum_gamerounds']
gate_40 = df[df['version'] == 'gate_40']['sum_gamerounds']

print("=== Normality Check: sum_gamerounds ===")
print("\ngate_30:")
norm_30 = check_normality(gate_30)
for k, v in norm_30.items():
    print(f"  {k:12s}: {v}")

print("\ngate_40:")
norm_40 = check_normality(gate_40)
for k, v in norm_40.items():
    print(f"  {k:12s}: {v}")

ab_test_framework.py loaded successfully.
=== Normality Check: sum_gamerounds ===

gate_30:
  is_normal   : False
  test_used   : dagostino
  statistic   : 5071.98946
  p_value     : 0.0

gate_40:
  is_normal   : False
  test_used   : dagostino
  statistic   : 4919.266325
  p_value     : 0.0
=== Normality Check: sum_gamerounds ===

gate_30:
  is_normal   : False
  test_used   : dagostino
  statistic   : 5071.98946
  p_value     : 0.0

gate_40:
  is_normal   : False
  test_used   : dagostino
  statistic   : 4919.266325
  p_value     : 0.0


In [14]:
# --- Reload ---
importlib.reload(ab_test_framework)
from ab_test_framework import *

# --- Run continuous test on sum_gamerounds ---
gate_30 = df[df['version'] == 'gate_30']['sum_gamerounds']
gate_40 = df[df['version'] == 'gate_40']['sum_gamerounds']

result_continuous = run_continuous_test(gate_30, gate_40)

print("=== Continuous Test Result: sum_gamerounds ===")
for k, v in result_continuous.items():
    if k not in ('normality_a', 'normality_b'):
        print(f"  {k:22s}: {v}")

print("\n--- Normality Details ---")
print(f"  group_a normal: {result_continuous['normality_a']['is_normal']}")
print(f"  group_b normal: {result_continuous['normality_b']['is_normal']}")

ab_test_framework.py loaded successfully.
=== Continuous Test Result: sum_gamerounds ===
  outcome_type          : continuous
  test_used             : mann_whitney_u
  statistic             : 1024285761.5
  p_value               : 0.050892
  effect_size           : -0.007504
  effect_size_type      : rank_biserial
  effect_magnitude      : negligible
  confidence_interval   : (np.float64(0.0), np.float64(1.0))
  significant           : False

--- Normality Details ---
  group_a normal: False
  group_b normal: False


In [15]:
# --- Reload ---
importlib.reload(ab_test_framework)
from ab_test_framework import *

# --- Test categorical branch on retention_1 and retention_7 ---
for outcome in ['retention_1', 'retention_7']:
    result = run_categorical_test(
        data        = df,
        group_col   = 'version',
        outcome_col = outcome,
        group_a     = 'gate_30',
        group_b     = 'gate_40'
    )
    print(f"=== Categorical Test Result: {outcome} ===")
    for k, v in result.items():
        if k != 'contingency_table':
            print(f"  {k:22s}: {v}")
    print()

ab_test_framework.py loaded successfully.
=== Categorical Test Result: retention_1 ===
  outcome_type          : categorical
  test_used             : chi_square
  statistic             : 3.1698
  p_value               : 0.07501
  degrees_of_freedom    : 1
  effect_size           : 0.005928
  effect_size_type      : cramers_v
  effect_magnitude      : negligible
  prop_a                : 0.448198
  prop_b                : 0.442283
  prop_diff             : 0.005915
  ci_group_a            : (np.float64(0.443592), np.float64(0.452812))
  ci_group_b            : (np.float64(0.437724), np.float64(0.446851))
  confidence_interval   : (np.float64(-0.000572), np.float64(0.012403))
  significant           : False

=== Categorical Test Result: retention_7 ===
  outcome_type          : categorical
  test_used             : chi_square
  statistic             : 9.9153
  p_value               : 0.001639
  degrees_of_freedom    : 1
  effect_size           : 0.010485
  effect_size_type      : cramer

In [16]:
# --- Reload ---
importlib.reload(ab_test_framework)
from ab_test_framework import *

# --- Test verdict generator on all three outcomes ---

# Continuous
gate_30 = df[df['version'] == 'gate_30']['sum_gamerounds']
gate_40 = df[df['version'] == 'gate_40']['sum_gamerounds']
result_cont = run_continuous_test(gate_30, gate_40)
verdict_cont = generate_verdict(
    result_cont, 'gate_30', 'gate_40', 'sum_gamerounds', ALPHA_DEFAULT
)

# Categorical
result_r1 = run_categorical_test(df, 'version', 'retention_1', 'gate_30', 'gate_40')
verdict_r1 = generate_verdict(
    result_r1, 'gate_30', 'gate_40', 'retention_1', ALPHA_DEFAULT
)

result_r7 = run_categorical_test(df, 'version', 'retention_7', 'gate_30', 'gate_40')
verdict_r7 = generate_verdict(
    result_r7, 'gate_30', 'gate_40', 'retention_7', ALPHA_DEFAULT
)

print("=== Plain Language Verdicts ===\n")
print(verdict_cont)
print()
print(verdict_r1)
print()
print(verdict_r7)

ab_test_framework.py loaded successfully.
=== Plain Language Verdicts ===

[sum_gamerounds] Using a Mann-Whitney U test (non-normal distribution detected), the difference between gate_30 and gate_40 was NOT statistically significant (p=0.0509, alpha=0.05). Effect size was negligible (rank_biserial=-0.0075). 95% bootstrap CI for median difference: [0.00, 1.00].

[retention_1] Using a Chi-square test (binary/categorical outcome), the difference between gate_30 and gate_40 was NOT statistically significant (p=0.0750, alpha=0.05). Effect size was negligible (cramers_v=0.0059). gate_30 rate: 44.82%, gate_40 rate: 44.23% (difference: 0.59 pp).

[retention_7] Using a Chi-square test (binary/categorical outcome), the difference between gate_30 and gate_40 was STATISTICALLY SIGNIFICANT (p=0.0016, alpha=0.05). Effect size was negligible (cramers_v=0.0105). gate_30 rate: 19.02%, gate_40 rate: 18.20% (difference: 0.82 pp).


In [17]:
# --- Reload ---
importlib.reload(ab_test_framework)
from ab_test_framework import *

# --- Run the full framework on all three outcomes ---
outcomes = ['sum_gamerounds', 'retention_1', 'retention_7']

print("=== Full Framework Validation Against Phase 2 ===\n")

for outcome in outcomes:
    result = run_ab_test(
        data        = df,
        group_col   = 'version',
        outcome_col = outcome,
        group_a     = 'gate_30',
        group_b     = 'gate_40'
    )
    print(f"Outcome     : {outcome}")
    print(f"Test used   : {result['test_used']}")
    print(f"p-value     : {result['p_value']}")
    print(f"Significant : {result['significant']}")
    print(f"Effect size : {result['effect_size']} ({result['effect_magnitude']})")
    print(f"Verdict     : {result['verdict']}")
    print("-" * 70)

ab_test_framework.py loaded successfully.
=== Full Framework Validation Against Phase 2 ===

Outcome     : sum_gamerounds
Test used   : mann_whitney_u
p-value     : 0.050892
Significant : False
Effect size : -0.007504 (negligible)
Verdict     : [sum_gamerounds] Using a Mann-Whitney U test (non-normal distribution detected), the difference between gate_30 and gate_40 was NOT statistically significant (p=0.0509, alpha=0.05). Effect size was negligible (rank_biserial=-0.0075). 95% bootstrap CI for median difference: [0.00, 1.00].
----------------------------------------------------------------------
Outcome     : retention_1
Test used   : chi_square
p-value     : 0.07501
Significant : False
Effect size : 0.005928 (negligible)
Verdict     : [retention_1] Using a Chi-square test (binary/categorical outcome), the difference between gate_30 and gate_40 was NOT statistically significant (p=0.0750, alpha=0.05). Effect size was negligible (cramers_v=0.0059). gate_30 rate: 44.82%, gate_40 rate: 4

In [18]:
# --- Reload ---
importlib.reload(ab_test_framework)
from ab_test_framework import *

# --- Run the full framework on all three outcomes ---
outcomes = ['sum_gamerounds', 'retention_1', 'retention_7']

print("=== Full Framework Validation Against Phase 2 ===\n")

for outcome in outcomes:
    result = run_ab_test(
        data        = df,
        group_col   = 'version',
        outcome_col = outcome,
        group_a     = 'gate_30',
        group_b     = 'gate_40'
    )
    print(f"Outcome     : {outcome}")
    print(f"Test used   : {result['test_used']}")
    print(f"p-value     : {result['p_value']}")
    print(f"Significant : {result['significant']}")
    print(f"Effect size : {result['effect_size']} ({result['effect_magnitude']})")
    print(f"Verdict     : {result['verdict']}")
    print("-" * 70)

ab_test_framework.py loaded successfully.
=== Full Framework Validation Against Phase 2 ===

Outcome     : sum_gamerounds
Test used   : mann_whitney_u
p-value     : 0.050892
Significant : False
Effect size : -0.007504 (negligible)
Verdict     : [sum_gamerounds] Using a Mann-Whitney U test (non-normal distribution detected), the difference between gate_30 and gate_40 was NOT statistically significant (p=0.0509, alpha=0.05). Effect size was negligible (rank_biserial=-0.0075). 95% bootstrap CI for median difference: [0.00, 1.00].
----------------------------------------------------------------------
Outcome     : retention_1
Test used   : chi_square
p-value     : 0.07501
Significant : False
Effect size : 0.005928 (negligible)
Verdict     : [retention_1] Using a Chi-square test (binary/categorical outcome), the difference between gate_30 and gate_40 was NOT statistically significant (p=0.0750, alpha=0.05). Effect size was negligible (cramers_v=0.0059). gate_30 rate: 44.82%, gate_40 rate: 4

In [19]:
# --- Reload ---
importlib.reload(ab_test_framework)
from ab_test_framework import *

import numpy as np

# --- Generate synthetic normal data ---
np.random.seed(42)

# Two groups drawn from normal distributions
# Group A: mean=100, std=15, n=200
# Group B: mean=110, std=15, n=200
# True difference = 10 units — should be significant
synthetic_df = pd.DataFrame({
    'group'   : ['A'] * 200 + ['B'] * 200,
    'outcome' : np.concatenate([
        np.random.normal(loc=100, scale=15, size=200),
        np.random.normal(loc=110, scale=15, size=200)
    ])
})

print("=== Synthetic Data Summary ===")
print(synthetic_df.groupby('group')['outcome'].describe().round(2))

# --- Run framework ---
result_synthetic = run_ab_test(
    data        = synthetic_df,
    group_col   = 'group',
    outcome_col = 'outcome',
    group_a     = 'A',
    group_b     = 'B'
)

print("\n=== Framework Result on Synthetic Normal Data ===")
print(f"Outcome type  : {result_synthetic['outcome_type']}")
print(f"Test used     : {result_synthetic['test_used']}")
print(f"Normal A?     : {result_synthetic['normality_a']['is_normal']}")
print(f"Normal B?     : {result_synthetic['normality_b']['is_normal']}")
print(f"Statistic     : {result_synthetic['statistic']}")
print(f"p-value       : {result_synthetic['p_value']}")
print(f"Significant   : {result_synthetic['significant']}")
print(f"Effect size   : {result_synthetic['effect_size']} ({result_synthetic['effect_magnitude']})")
print(f"\nVerdict:\n{result_synthetic['verdict']}")

ab_test_framework.py loaded successfully.
=== Synthetic Data Summary ===
       count    mean    std    min     25%     50%     75%     max
group                                                             
A      200.0   99.39  13.97  60.70   89.42   99.94  107.51  140.80
B      200.0  111.29  14.81  61.38  100.91  111.18  120.31  167.79

=== Framework Result on Synthetic Normal Data ===
Outcome type  : continuous
Test used     : independent_samples_ttest
Normal A?     : True
Normal B?     : True
Statistic     : -8.2687
p-value       : 0.0
Significant   : True
Effect size   : -0.826866 (large)

Verdict:
[outcome] Using a independent samples t-test (normal distribution confirmed), the difference between A and B was STATISTICALLY SIGNIFICANT (p=0.0000, alpha=0.05). Effect size was large (cohens_d=-0.8269). 95% bootstrap CI for median difference: [-14.95, -7.74].


In [20]:
# --- Reload ---
importlib.reload(ab_test_framework)
from ab_test_framework import *

print("=== Input Validation Tests ===\n")

# --- Test 1: Wrong group_col ---
print("Test 1: Wrong group_col name")
try:
    run_ab_test(df, 'wrong_col', 'retention_7', 'gate_30', 'gate_40')
except ValueError as e:
    print(f"  Caught ValueError: {e}\n")

# --- Test 2: Wrong outcome_col ---
print("Test 2: Wrong outcome_col name")
try:
    run_ab_test(df, 'version', 'wrong_outcome', 'gate_30', 'gate_40')
except ValueError as e:
    print(f"  Caught ValueError: {e}\n")

# --- Test 3: Wrong group label ---
print("Test 3: Wrong group label")
try:
    run_ab_test(df, 'version', 'retention_7', 'gate_30', 'gate_99')
except ValueError as e:
    print(f"  Caught ValueError: {e}\n")

# --- Test 4: Custom alpha ---
print("Test 4: Custom alpha = 0.01")
result = run_ab_test(
    data        = df,
    group_col   = 'version',
    outcome_col = 'retention_7',
    group_a     = 'gate_30',
    group_b     = 'gate_40',
    alpha       = 0.01
)
print(f"  p-value    : {result['p_value']}")
print(f"  alpha used : {result['alpha']}")
print(f"  Significant: {result['significant']}")
print(f"  Verdict    : {result['verdict'][:80]}...")

ab_test_framework.py loaded successfully.
=== Input Validation Tests ===

Test 1: Wrong group_col name
  Caught ValueError: group_col 'wrong_col' not found in data.

Test 2: Wrong outcome_col name
  Caught ValueError: outcome_col 'wrong_outcome' not found in data.

Test 3: Wrong group label
  Caught ValueError: group_b 'gate_99' not found in version.

Test 4: Custom alpha = 0.01
  p-value    : 0.001639
  alpha used : 0.01
  Significant: True
  Verdict    : [retention_7] Using a Chi-square test (binary/categorical outcome), the differen...


In [21]:
result = run_ab_test(
    data        = df,
    group_col   = 'version',
    outcome_col = 'retention_7',
    group_a     = 'gate_30',
    group_b     = 'gate_40',
    alpha       = 0.01
)
print(result['verdict'])

[retention_7] Using a Chi-square test (binary/categorical outcome), the difference between gate_30 and gate_40 was STATISTICALLY SIGNIFICANT (p=0.0016, alpha=0.01). Effect size was negligible (cramers_v=0.0105). gate_30 rate: 19.02%, gate_40 rate: 18.20% (difference: 0.82 pp).
